# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iammns9/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis
One row represents the daily performance of a single content page (`content_hash_id`) for a single client (`client_hash_id`) on one reporting date (`report_date`).

## Time Window
I use March 2026 as the development month for verification and feature engineering. I deliberately exclude the final month (June 2026) because it is reserved as the sealed test period.

In [29]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

1. gsc_impressions
   - Number of Google Search impressions.

2. gsc_clicks
   - Number of Google Search clicks.

3. gsc_sum_position
   - Sum of Google Search positions.

4. gsc_avg_position
   - Average Google Search position.

5. ga4_sessions
   - Number of GA4 sessions.

## Label

Whether the content receives at least one Google Search click during the prediction period (derived from `gsc_clicks > 0` for this demonstration).

## Context

- report_date
- client_hash_id
- content_hash_id

These identify the observation but are not used as predictive features.

## Excluded

- gsc_data_available
- ga4_data_available
- month
- Future outcome or label-derived columns (excluded to prevent data leakage).

These are excluded because they are availability indicators, filtering fields, or would introduce data leakage.

In [30]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 1
""").df().columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification 1 — Grain

This query confirms that each row represents one unique combination of report date, client, and content page.

In [31]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_rows
FROM {TABLES['fact_daily']}
WHERE year(report_date)=2026
AND month(report_date)=3
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,9841378,9841378


### Verification 2 — Row Count and Date Window

This query confirms the number of rows and verifies that the selected development window covers March 2026.

In [32]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE year(report_date)=2026
AND month(report_date)=3
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


### Verification 3 — Data Availability

This query checks how many rows remain after filtering for available Google Search Console data using `gsc_data_available IS TRUE`.

In [33]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE year(report_date)=2026
AND month(report_date)=3
AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## Five Feature Frame

The following feature frame is built from March 2026 and contains five features that are available at the decision moment.

| Feature | Available when? |
|---------|-----------------|
| gsc_impressions | Knowable at the decision moment because it is already recorded for the reporting day. |
| gsc_clicks | Knowable at the decision moment because daily click counts are already available. |
| gsc_sum_position | Knowable at the decision moment because it is calculated from observed search performance. |
| gsc_avg_position | Knowable at the decision moment because average ranking is already measured. |
| ga4_sessions | Knowable at the decision moment because GA4 session data has already been collected. |

In [34]:
feature_frame = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    gsc_avg_position,
    ga4_sessions
FROM {TABLES['fact_daily']}
WHERE year(report_date)=2026
AND month(report_date)=3
AND gsc_data_available IS TRUE
LIMIT 10
""").df()

feature_frame

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67,3.350000,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0,0.000000,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616,4.928000,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28,4.000000,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25,2.272727,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,1756,7.347280,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,1496,7.832461,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,180,3.272727,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,434,5.636364,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,9,4.500000,<NA>


## Leakage Experiment

An honest model was trained using only valid features.

Then a label-derived feature was intentionally added to demonstrate data leakage. The model accuracy increased unrealistically, showing why label-derived information must never be included during training.

**Honest Accuracy:** 0.8748

**Leaked Accuracy:** 1.0000

In [35]:
leak_df = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    gsc_avg_position,
    ga4_sessions,
    CASE
        WHEN gsc_clicks > 0 THEN 1
        ELSE 0
    END AS label
FROM {TABLES['fact_daily']}
WHERE year(report_date)=2026
AND month(report_date)=3
AND gsc_data_available IS TRUE
LIMIT 50000
""").df()

leak_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_sessions,label
0,20,0,67,3.350000,<NA>,0
1,1,0,0,0.000000,<NA>,0
2,125,1,616,4.928000,<NA>,1
3,7,0,28,4.000000,<NA>,0
4,11,0,25,2.272727,<NA>,0


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = [
    "gsc_impressions",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_sessions"
]

X = leak_df[features].fillna(0)
y = leak_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest Accuracy:", accuracy_score(y_test, pred))

Honest Accuracy: 0.8748


In [37]:
features_with_leak = [
    "gsc_impressions",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_sessions",
    "label"
]

X = leak_df[features_with_leak].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Leaked Accuracy:", accuracy_score(y_test, pred))

Leaked Accuracy: 1.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- The analysis uses only March 2026 as the development period.
- History is unbalanced because different clients have different amounts of historical data.
- Many rows contain only GSC data, while GA4 metrics are unavailable.
- Client and content identifiers are anonymized, so business-specific interpretation is not possible.
- June 2026 is excluded because it is the sealed test month.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.